In [ ]:
   import os
   os.makedirs("/home/ibm-user/private", exist_ok=True)
   print(os.listdir("/home/ibm-user/shared"))

In [ ]:
import pandas as pd, numpy as np

RAW = "/home/ibm-user/private/HI-Small_Trans.csv"
CH = 500_000
cols = ["From Bank", "Account", "To Bank", "Account.1", "Is Laundering"]
dt = {"Account": str, "Account.1": str}

# Pass 1: laundering-linked accounts, and the list of all accounts
ring, allacc = set(), set()
for ch in pd.read_csv(RAW, chunksize=CH, usecols=cols, dtype=dt):
    fk = ch["From Bank"].astype(str) + "|" + ch["Account"]
    tk = ch["To Bank"].astype(str) + "|" + ch["Account.1"]
    allacc |= set(fk); allacc |= set(tk)
    L = ch["Is Laundering"] == 1
    ring |= set(fk[L]); ring |= set(tk[L])

# A random set of normal accounts, kept with their FULL history
rng = np.random.default_rng(42)
others = np.array(sorted(allacc - ring))
extra = set(rng.choice(others, size=15_000, replace=False))
keep = ring | extra
print("laundering-linked:", len(ring), "| normal sampled:", len(extra))

# Pass 2: keep every transaction that touches a kept account
parts = []
for ch in pd.read_csv(RAW, chunksize=CH, dtype=dt):
    fk = ch["From Bank"].astype(str) + "|" + ch["Account"]
    tk = ch["To Bank"].astype(str) + "|" + ch["Account.1"]
    parts.append(ch[fk.isin(keep) | tk.isin(keep)])
df = pd.concat(parts, ignore_index=True)

self_tx = (df["From Bank"] == df["To Bank"]) & (df["Account"] == df["Account.1"])
df = df[~self_tx | (df["Is Laundering"] == 1)]
df = df.sort_values("Timestamp").reset_index(drop=True)

print(df.shape)
print(df["Is Laundering"].value_counts())
print(df["Timestamp"].min(), df["Timestamp"].max())

In [ ]:
import os, getpass, hmac, hashlib

# Secret key: stored outside shared, never printed
KEY_PATH = "/home/ibm-user/private/gh_key.txt"
if not os.path.exists(KEY_PATH):
    with open(KEY_PATH, "w") as f:
        f.write(getpass.getpass("Enter team secret key (20+ characters): "))
    os.chmod(KEY_PATH, 0o600)
KEY = open(KEY_PATH).read().strip().encode()

# Rename to the agreed columns
d = df.rename(columns={
    "Timestamp": "timestamp", "From Bank": "from_bank_raw", "Account": "from_raw",
    "To Bank": "to_bank_raw", "Account.1": "to_raw", "Amount Paid": "amount_paid",
    "Payment Currency": "currency", "Payment Format": "payment_format",
    "Is Laundering": "is_laundering"})
d["timestamp"] = pd.to_datetime(d["timestamp"], format="%Y/%m/%d %H:%M")
d["tx_id"] = range(len(d))

# Pseudonymise: hash each unique account once, keyed so only we can reproduce codes
def pseudo(bank, acct):
    return "A_" + hmac.new(KEY, f"{bank}|{acct}".encode(), hashlib.sha256).hexdigest()[:12]

d["from_key"] = d["from_bank_raw"].astype(str) + "|" + d["from_raw"].astype(str)
d["to_key"]   = d["to_bank_raw"].astype(str)   + "|" + d["to_raw"].astype(str)
lookup = {k: pseudo(*k.split("|", 1)) for k in pd.unique(pd.concat([d["from_key"], d["to_key"]]))}
d["from_acct"] = d["from_key"].map(lookup)
d["to_acct"]   = d["to_key"].map(lookup)

# Three demo banks
banks = sorted(pd.unique(pd.concat([d["from_bank_raw"], d["to_bank_raw"]])))
bank_label = {b: f"Bank_{(i % 3) + 1}" for i, b in enumerate(banks)}
d["from_bank"] = d["from_bank_raw"].map(bank_label)
d["to_bank"]   = d["to_bank_raw"].map(bank_label)
print(d["from_bank"].value_counts())
print("cross-bank share:", (d["from_bank"] != d["to_bank"]).mean())

# Save clean files for the team
out_cols = ["tx_id","timestamp","from_acct","to_acct","from_bank","to_bank",
            "amount_paid","currency","payment_format","is_laundering"]
clean = d[out_cols]
clean.to_csv("/home/ibm-user/shared/clean_transactions.csv", index=False)
clean.sample(20_000, random_state=1).sort_values("timestamp").to_csv(
    "/home/ibm-user/shared/clean_transactions_sample.csv", index=False)

# Private lookup: only the owning bank could reverse a code (kept OUTSIDE shared)
rev = pd.DataFrame({"code": list(lookup.values()), "real_key": list(lookup.keys())})
rev["owner_bank"] = rev["real_key"].str.split("|").str[0].astype(int).map(bank_label)
rev.to_csv("/home/ibm-user/private/code_to_account.csv", index=False)

# Checks
c = pd.read_csv("/home/ibm-user/shared/clean_transactions.csv")
assert list(c.columns) == out_cols
assert c["tx_id"].is_unique and not c.isna().any().any()
raw_ids = set(df["Account"].astype(str)) | set(df["Account.1"].astype(str))
assert not (set(c["from_acct"]) | set(c["to_acct"])) & raw_ids, "real IDs leaked!"
assert d.groupby("from_key")["from_acct"].nunique().max() == 1
assert len(set(lookup.values())) == len(lookup)
print(c.shape, "laundering rate:", round(c["is_laundering"].mean(), 4))
print("all checks passed")

In [ ]:
import pandas as pd, numpy as np, networkx as nx

c = pd.read_csv("/home/ibm-user/shared/clean_transactions.csv", parse_dates=["timestamp"])

# 1. Per-account counts and totals (amounts are mixed currencies)
out_ = c.groupby("from_acct").agg(tx_out=("tx_id", "size"),
                                  n_receivers=("to_acct", "nunique"),
                                  total_out=("amount_paid", "sum"))
in_ = c.groupby("to_acct").agg(tx_in=("tx_id", "size"),
                               n_senders=("from_acct", "nunique"),
                               total_in=("amount_paid", "sum"))
f = out_.join(in_, how="outer").fillna(0)
f.index.name = "acct"
f["out_in_ratio"] = f["total_out"] / (f["total_in"] + 1)

# 2. PageRank on the aggregated network (ignoring self-loops)
e = c.groupby(["from_acct", "to_acct"], as_index=False)["amount_paid"].sum()
e = e[e["from_acct"] != e["to_acct"]]
G = nx.from_pandas_edgelist(e, "from_acct", "to_acct", create_using=nx.DiGraph)
f["pagerank"] = pd.Series(nx.pagerank(G, alpha=0.85))

# 3. Money going round: accounts with a two-way link (A->B and B->A)
rev = e[["from_acct", "to_acct"]].rename(columns={"from_acct": "to_acct", "to_acct": "from_acct"})
rec = e[["from_acct", "to_acct"]].merge(rev, on=["from_acct", "to_acct"])
f["has_reciprocal"] = f.index.isin(set(rec["from_acct"])).astype(int)

# 4. Pass-through speed: how soon after receiving money does the account send money on?
outs = c[["from_acct", "timestamp"]].rename(columns={"from_acct": "acct", "timestamp": "t_out"}).sort_values("t_out")
ins = c[["to_acct", "timestamp"]].rename(columns={"to_acct": "acct", "timestamp": "t_in"}).sort_values("t_in")
m = pd.merge_asof(outs, ins, left_on="t_out", right_on="t_in", by="acct", direction="backward")
m["gap_min"] = (m["t_out"] - m["t_in"]).dt.total_seconds() / 60
m["fast"] = (m["gap_min"] <= 60).astype(int)
ff = m.groupby("acct").agg(fast_forward_share=("fast", "mean"), median_hold_min=("gap_min", "median"))
f = f.join(ff)
f["fast_forward_share"] = f["fast_forward_share"].fillna(0)
f["median_hold_min"] = f["median_hold_min"].fillna(-1)   # -1 = never forwarded money after receiving it

f = f.reset_index()
f.to_csv("/home/ibm-user/shared/graph_features.csv", index=False)
print(f.shape)
print(f.describe().T[["mean", "50%", "max"]])

In [ ]:
f = pd.read_csv("/home/ibm-user/shared/graph_features.csv")

for col in ["total_out", "total_in", "tx_out", "tx_in", "n_receivers", "n_senders"]:
    f["log_" + col] = np.log1p(f[col])
f["log_out_in_gap"] = f["log_total_out"] - f["log_total_in"]
f = f.drop(columns=["out_in_ratio"])
f.to_csv("/home/ibm-user/shared/graph_features.csv", index=False)

# Check only (not saved): do the features differ for laundering-linked accounts?
lab = (pd.concat([c[["from_acct", "is_laundering"]].rename(columns={"from_acct": "acct"}),
                  c[["to_acct", "is_laundering"]].rename(columns={"to_acct": "acct"})])
         .groupby("acct")["is_laundering"].max().rename("label"))
chk = f.merge(lab, on="acct")
print(chk["label"].value_counts())
print(chk.groupby("label")[["tx_out", "tx_in", "n_receivers", "n_senders",
                            "pagerank", "has_reciprocal", "fast_forward_share"]].mean().T)
print(chk.sort_values("tx_out", ascending=False).head(10)[["acct", "tx_out", "n_receivers", "label"]])

In [ ]:
import pandas as pd
c = pd.read_csv("/home/ibm-user/shared/clean_transactions.csv")

g = pd.concat([
    c[["from_acct", "is_laundering"]].rename(columns={"from_acct": "acct"}),
    c[["to_acct", "is_laundering"]].rename(columns={"to_acct": "acct"})])
lab = g.groupby("acct")["is_laundering"].agg(n_laundering_tx="sum", n_tx="size").reset_index()
lab["laundering_share"] = lab["n_laundering_tx"] / lab["n_tx"]
lab["touched_any"] = (lab["n_laundering_tx"] >= 1).astype(int)
# Stricter: at least 2 laundering transactions AND at least 5% of the account's activity
lab["is_laundering_strict"] = ((lab["n_laundering_tx"] >= 2) & (lab["laundering_share"] >= 0.05)).astype(int)
lab.to_csv("/home/ibm-user/shared/account_labels.csv", index=False)

print(lab[["touched_any", "is_laundering_strict"]].sum())
print(lab.sort_values("n_tx", ascending=False).head(10)[["acct", "n_tx", "n_laundering_tx", "laundering_share"]])

In [ ]:
print("accounts by laundering share (touched at least one laundering tx):")
for t in [0.01, 0.05, 0.10, 0.25, 0.50]:
    s = lab[(lab["touched_any"] == 1) & (lab["laundering_share"] >= t)]
    print(f"share >= {t:>4}: {len(s):5d} accounts | median tx: {s['n_tx'].median():.0f}")

print()
lost = lab[(lab["touched_any"] == 1) & (lab["is_laundering_strict"] == 0)]
print("accounts the strict rule drops:", len(lost))
print(lost["n_tx"].describe()[["50%", "75%", "max"]])
print("of those, share >= 0.25:", (lost["laundering_share"] >= 0.25).sum())

In [ ]:
lab["is_laundering_final"] = ((lab["n_laundering_tx"] >= 1) & (lab["laundering_share"] >= 0.05)).astype(int)
lab.to_csv("/home/ibm-user/shared/account_labels.csv", index=False)

print(lab["is_laundering_final"].sum(), "positive accounts out of", len(lab))
print(lab[lab["is_laundering_final"] == 1].sort_values("n_tx", ascending=False)
        .head(5)[["acct", "n_tx", "n_laundering_tx", "laundering_share"]])

In [ ]:
import pandas as pd, networkx as nx

c = pd.read_csv("/home/ibm-user/shared/clean_transactions.csv")
c["timestamp"] = pd.to_datetime(c["timestamp"])

acct_bank = dict(zip(c["from_acct"], c["from_bank"]))
acct_bank.update(dict(zip(c["to_acct"], c["to_bank"])))

L = c[(c["is_laundering"] == 1) & (c["from_acct"] != c["to_acct"])]
LG = nx.from_pandas_edgelist(L, "from_acct", "to_acct", create_using=nx.DiGraph)

rows, members = [], {}
for i, comp in enumerate(nx.weakly_connected_components(LG)):
    t = L[L["from_acct"].isin(comp) & L["to_acct"].isin(comp)]
    members[i] = comp
    rows.append(dict(comp=i, n_accts=len(comp), n_tx=len(t),
                     n_banks=len({acct_bank[a] for a in comp}),
                     has_cycle=not nx.is_directed_acyclic_graph(LG.subgraph(comp)),
                     hours=round((t["timestamp"].max() - t["timestamp"].min()).total_seconds() / 3600, 1)))
R = pd.DataFrame(rows)
print(len(R), "laundering groups found")

cand = (R[(R["n_accts"].between(6, 25)) & (R["n_banks"] == 3)]
        .sort_values(["has_cycle", "hours"], ascending=[False, True]))
print(cand.head(8).to_string(index=False))

if len(cand):
    best = cand.iloc[0]
    m = members[best["comp"]]
    ring = L[L["from_acct"].isin(m) & L["to_acct"].isin(m)].sort_values("timestamp").copy()
    ring["role"] = "ring"
    ctx = c[(c["from_acct"].isin(m) | c["to_acct"].isin(m)) & (c["is_laundering"] == 0)].copy()
    ctx["role"] = "normal"
    demo = pd.concat([ring, ctx]).sort_values("timestamp")
    demo.to_csv("/home/ibm-user/shared/demo_ring.csv", index=False)
    print("\nchosen group:", best.to_dict())
    print("normal-looking transactions around it:", len(ctx))
    print(ring[["timestamp", "from_acct", "to_acct", "from_bank", "to_bank",
                "amount_paid", "currency"]].to_string(index=False))
else:
    print("no group touches all 3 banks; tell me the table above and we'll relax the filter")

In [ ]:

import pandas as pd
import json
import html
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path("/home/ibm-user/shared")
demo_path = ROOT / "demo_ring.csv"

assert demo_path.exists(), "demo_ring.csv is missing. Run the ring-finder cell first."

demo = pd.read_csv(demo_path)
assert "role" in demo.columns, "The demo file needs a role column."

ring = demo[demo["role"] == "ring"].copy()
normal = demo[demo["role"] == "normal"].copy()

accounts = sorted(
    set(ring["from_acct"]) | set(ring["to_acct"])
)
banks = ["Bank_1", "Bank_2", "Bank_3"]

# Load teammate results if present.
freeze_path = ROOT / "freeze_plan_explained.csv"
if not freeze_path.exists():
    freeze_path = ROOT / "freeze_plan.csv"

freeze_rows = []
if freeze_path.exists():
    fp = pd.read_csv(freeze_path).fillna("")
    if "acct" in fp.columns:
        freeze_rows = fp[
            fp["acct"].isin(accounts)
        ].to_dict(orient="records")

# Do not show unverified model output as a confirmed result.
data = {
    "ring": ring.fillna("").to_dict(orient="records"),
    "normal": normal.fillna("").to_dict(orient="records"),
    "accounts": accounts,
    "banks": banks,
    "freezes": freeze_rows
}

payload = json.dumps(data, default=str).replace("<", "\\u003c")

page = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Golden Hour | Cross-Bank Detection</title>
<style>
* { box-sizing:border-box }
body { margin:0; padding:28px; background:#08111f; color:#e8eef8;
       font:15px system-ui,Arial,sans-serif }
header { display:flex; justify-content:space-between; flex-wrap:wrap;
         gap:14px; align-items:center; margin-bottom:26px }
h1 { margin:0; font-size:30px; color:#f8ca72 }
h2 { margin:0 0 14px; font-size:19px }
small,.muted { color:#9fb0c6 }
.grid { display:grid; grid-template-columns:repeat(3,minmax(0,1fr));
        gap:15px }
.panel { background:#132238; border:1px solid #30445c;
         border-radius:14px; padding:20px; min-width:0 }
button { border:0; border-radius:10px; padding:13px 20px;
         background:#e6ad51; color:#151a22; font-weight:750;
         cursor:pointer; font-size:14px }
button.active { background:#f87171 }
table { border-collapse:collapse; width:100%; font-size:12px }
th,td { padding:9px 6px; border-bottom:1px solid #30445c;
        text-align:left; overflow-wrap:anywhere }
th { color:#aac0d8 }
tr.flagged { background:#683333 }
.stat { font-size:30px; font-weight:800; color:#f8ca72 }
.section { margin-top:20px }
#network { width:100%; max-width:750px; height:auto; display:block;
           margin:0 auto }
.nodeText { fill:white; font-size:11px; font-weight:650 }
.note { color:#abc0d5; font-size:13px; line-height:1.6 }
@media(max-width:900px) { .grid { grid-template-columns:1fr } }
</style>
</head>
<body>
<header>
  <div>
    <h1>GOLDEN HOUR</h1>
    <small>Privacy-aware cross-bank AML investigation</small>
  </div>
  <button id="toggle" onclick="toggleReveal()">Reveal Cross-Bank Network</button>
</header>

<div class="grid" id="banks"></div>

<section class="panel section">
  <h2 id="networkTitle">Shared intelligence — currently hidden</h2>
  <div id="summary" class="note">
    The banks are being examined independently.
  </div>
  <svg id="network" viewBox="0 0 750 490" style="display:none"></svg>
</section>

<section class="panel section" id="freezePanel" style="display:none">
  <h2>Investigator review</h2>
  <div id="freezeContent"></div>
</section>

<section class="section note">
  Demonstration using synthetic IBM AML transactions. The three banks
  are artificial groupings of source banks. Account IDs are
  pseudonymised, not anonymous. Highlighted transactions have
  dataset laundering labels; they are not proof of criminal guilt.
  No account is automatically frozen.
</section>

<script>
const DATA = __PAYLOAD__;
let revealed = false;
const colours = {
  Bank_1:"#a78bfa",
  Bank_2:"#f0b65d",
  Bank_3:"#4ac9dd"
};

function safe(v) {
  return String(v ?? "").replace(/[&<>"']/g, x =>
    ({"&":"&amp;","<":"&lt;",">":"&gt;",
      '"':"&quot;","'":"&#39;"}[x]));
}

function toggleReveal() {
  revealed = !revealed;
  const btn = document.getElementById("toggle");
  btn.textContent = revealed
    ? "Hide Cross-Bank Network"
    : "Reveal Cross-Bank Network";
  btn.classList.toggle("active", revealed);
  draw();
}

function draw() {
  const container = document.getElementById("banks");
  container.innerHTML = "";

  DATA.banks.forEach(bank => {
    const rows = [...DATA.normal,...DATA.ring]
      .filter(t => t.from_bank === bank || t.to_bank === bank)
      .sort((a,b) => String(a.timestamp).localeCompare(String(b.timestamp)));

    const shown = rows.slice(0, 16);
    const flagged = rows.filter(t=>t.role==="ring").length;

    const panel = document.createElement("div");
    panel.className = "panel";
    panel.innerHTML = `
      <h2 style="color:${colours[bank]}">${bank}</h2>
      <div class="stat">${rows.length}</div>
      <small>Transactions in the selected context</small>
      <p class="note">${
        revealed ? flagged+" labelled suspicious transactions visible"
                 : "Independent bank view — network labels hidden"
      }</p>
      <table>
        <thead><tr><th>Time</th><th>Account</th><th>Amount</th></tr></thead>
        <tbody>${shown.map(t=>`
          <tr class="${revealed && t.role==="ring" ? "flagged":""}">
            <td>${safe(String(t.timestamp).slice(5,16))}</td>
            <td>${safe(
              (t.from_bank===bank?t.from_acct:t.to_acct).slice(0,10)
            )}…</td>
            <td>${safe(Number(t.amount_paid).toLocaleString())}
              <small>${safe(t.currency)}</small></td>
          </tr>`).join("")}</tbody>
      </table>
      <p class="note">Showing ${shown.length} of ${rows.length} rows.
      A cross-bank transfer can appear on two bank screens.</p>`;
    container.appendChild(panel);
  });

  document.getElementById("networkTitle").textContent =
    revealed ? "Cross-bank laundering pattern revealed"
             : "Shared intelligence — currently hidden";

  document.getElementById("summary").textContent =
    revealed
      ? DATA.accounts.length+" linked accounts and "+DATA.ring.length+
        " labelled transactions across three demo banks."
      : "Each bank sees only its own slice of the transactions.";

  const svg = document.getElementById("network");
  svg.style.display = revealed ? "block" : "none";
  if (revealed) drawNetwork(svg);
  drawFreezes();
}

function drawNetwork(svg) {
  svg.innerHTML = "";
  const NS = "http://www.w3.org/2000/svg";

  function element(tag,attributes) {
    let e = document.createElementNS(NS,tag);
    Object.entries(attributes).forEach(([k,v])=>e.setAttribute(k,v));
    svg.appendChild(e);
    return e;
  }

  const bankOf = {};
  DATA.ring.forEach(t=>{
    bankOf[t.from_acct]=t.from_bank;
    bankOf[t.to_acct]=t.to_bank;
  });

  const pos = {};
  DATA.accounts.forEach((a,i)=>{
    let angle = -Math.PI/2 + 2*Math.PI*i/DATA.accounts.length;
    pos[a]={x:375+220*Math.cos(angle),y:245+170*Math.sin(angle)};
  });

  DATA.ring.forEach(t=>{
    const p=pos[t.from_acct], q=pos[t.to_acct];
    if (!p||!q) return;
    const dx=q.x-p.x,dy=q.y-p.y;
    const len=Math.hypot(dx,dy)||1;
    const ux=dx/len,uy=dy/len;
    const x1=p.x+24*ux,y1=p.y+24*uy;
    const x2=q.x-30*ux,y2=q.y-30*uy;
    element("line",{x1,y1,x2,y2,stroke:"#f8ca72",
                    "stroke-width":2.5});
    element("polygon",{
      points:`${x2},${y2} ${x2-12*ux-6*uy},${y2-12*uy+6*ux} ${x2-12*ux+6*uy},${y2-12*uy-6*ux}`,
      fill:"#f8ca72"
    });
  });

  DATA.accounts.forEach((a,i)=>{
    const p=pos[a];
    element("circle",{cx:p.x,cy:p.y,r:26,
                      fill:colours[bankOf[a]]||"#64748b"});
    const n=element("text",{x:p.x,y:p.y+4,"text-anchor":"middle",
                            class:"nodeText"});
    n.textContent=i+1;
    const lab=element("text",{x:p.x,y:p.y+42,
                             "text-anchor":"middle",
                             fill:"#dce7f4","font-size":11});
    lab.textContent=a.slice(0,10)+"…";
  });
}

function drawFreezes() {
  const panel=document.getElementById("freezePanel");
  panel.style.display=revealed?"block":"none";
  const el=document.getElementById("freezeContent");
  if (!DATA.freezes.length) {
    el.innerHTML=`<p class="note">No matching account from this ring
      was found in the current freeze plan. This does not mean the
      optimiser approved or rejected these accounts.</p>`;
    return;
  }
  el.innerHTML=DATA.freezes.map(f=>`
    <div style="border-bottom:1px solid #30445c;padding:12px 0">
      <strong>${safe(f.acct)}</strong>
      <p class="note">Risk score: ${safe(f.risk_score ?? "Unavailable")}</p>
      <p>${safe(f.explanation || f.reason_facts || "No explanation available")}</p>
    </div>`).join("")+
    `<p class="note">Recommendations only. Human approval required.</p>`;
}

draw();
</script>
</body>
</html>
"""

page = page.replace("__PAYLOAD__", payload)

out = ROOT / "golden_hour_demo.html"
out.write_text(page, encoding="utf-8")

print("Demo page created:", out)
print("Flagged transactions:", len(ring))
print("Normal context transactions:", len(normal))
print("Ring accounts:", len(accounts))
print("Matching freeze recommendations:", len(freeze_rows))
print("Open golden_hour_demo.html from the Jupyter file browser.")


In [ ]:

import pandas as pd
import json
import html
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path("/home/ibm-user/shared")
demo_path = ROOT / "demo_ring.csv"

assert demo_path.exists(), "demo_ring.csv is missing. Run the ring-finder cell first."

demo = pd.read_csv(demo_path)
assert "role" in demo.columns, "The demo file needs a role column."

ring = demo[demo["role"] == "ring"].copy()
normal = demo[demo["role"] == "normal"].copy()

accounts = sorted(
    set(ring["from_acct"]) | set(ring["to_acct"])
)
banks = ["Bank_1", "Bank_2", "Bank_3"]

# Load teammate results if present.
freeze_path = ROOT / "freeze_plan_explained.csv"
if not freeze_path.exists():
    freeze_path = ROOT / "freeze_plan.csv"

freeze_rows = []
if freeze_path.exists():
    fp = pd.read_csv(freeze_path).fillna("")
    if "acct" in fp.columns:
        freeze_rows = fp[
            fp["acct"].isin(accounts)
        ].to_dict(orient="records")

# Do not show unverified model output as a confirmed result.
data = {
    "ring": ring.fillna("").to_dict(orient="records"),
    "normal": normal.fillna("").to_dict(orient="records"),
    "accounts": accounts,
    "banks": banks,
    "freezes": freeze_rows
}

payload = json.dumps(data, default=str).replace("<", "\\u003c")

page = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Golden Hour | Cross-Bank Detection</title>
<style>
* { box-sizing:border-box }
body { margin:0; padding:28px; background:#08111f; color:#e8eef8;
       font:15px system-ui,Arial,sans-serif }
header { display:flex; justify-content:space-between; flex-wrap:wrap;
         gap:14px; align-items:center; margin-bottom:26px }
h1 { margin:0; font-size:30px; color:#f8ca72 }
h2 { margin:0 0 14px; font-size:19px }
small,.muted { color:#9fb0c6 }
.grid { display:grid; grid-template-columns:repeat(3,minmax(0,1fr));
        gap:15px }
.panel { background:#132238; border:1px solid #30445c;
         border-radius:14px; padding:20px; min-width:0 }
button { border:0; border-radius:10px; padding:13px 20px;
         background:#e6ad51; color:#151a22; font-weight:750;
         cursor:pointer; font-size:14px }
button.active { background:#f87171 }
table { border-collapse:collapse; width:100%; font-size:12px }
th,td { padding:9px 6px; border-bottom:1px solid #30445c;
        text-align:left; overflow-wrap:anywhere }
th { color:#aac0d8 }
tr.flagged { background:#683333 }
.stat { font-size:30px; font-weight:800; color:#f8ca72 }
.section { margin-top:20px }
#network { width:100%; max-width:750px; height:auto; display:block;
           margin:0 auto }
.nodeText { fill:white; font-size:11px; font-weight:650 }
.note { color:#abc0d5; font-size:13px; line-height:1.6 }
@media(max-width:900px) { .grid { grid-template-columns:1fr } }
</style>
</head>
<body>
<header>
  <div>
    <h1>GOLDEN HOUR</h1>
    <small>Privacy-aware cross-bank AML investigation</small>
  </div>
  <button id="toggle" onclick="toggleReveal()">Reveal Cross-Bank Network</button>
</header>

<div class="grid" id="banks"></div>

<section class="panel section">
  <h2 id="networkTitle">Shared intelligence — currently hidden</h2>
  <div id="summary" class="note">
    The banks are being examined independently.
  </div>
  <svg id="network" viewBox="0 0 750 490" style="display:none"></svg>
</section>

<section class="panel section" id="freezePanel" style="display:none">
  <h2>Investigator review</h2>
  <div id="freezeContent"></div>
</section>

<section class="section note">
  Demonstration using synthetic IBM AML transactions. The three banks
  are artificial groupings of source banks. Account IDs are
  pseudonymised, not anonymous. Highlighted transactions have
  dataset laundering labels; they are not proof of criminal guilt.
  No account is automatically frozen.
</section>

<script>
const DATA = __PAYLOAD__;
let revealed = false;
const colours = {
  Bank_1:"#a78bfa",
  Bank_2:"#f0b65d",
  Bank_3:"#4ac9dd"
};

function safe(v) {
  return String(v ?? "").replace(/[&<>"']/g, x =>
    ({"&":"&amp;","<":"&lt;",">":"&gt;",
      '"':"&quot;","'":"&#39;"}[x]));
}

function toggleReveal() {
  revealed = !revealed;
  const btn = document.getElementById("toggle");
  btn.textContent = revealed
    ? "Hide Cross-Bank Network"
    : "Reveal Cross-Bank Network";
  btn.classList.toggle("active", revealed);
  draw();
}

function draw() {
  const container = document.getElementById("banks");
  container.innerHTML = "";

  DATA.banks.forEach(bank => {
    const rows = [...DATA.normal,...DATA.ring]
      .filter(t => t.from_bank === bank || t.to_bank === bank)
      .sort((a,b) => String(a.timestamp).localeCompare(String(b.timestamp)));

    const shown = rows.slice(0, 16);
    const flagged = rows.filter(t=>t.role==="ring").length;

    const panel = document.createElement("div");
    panel.className = "panel";
    panel.innerHTML = `
      <h2 style="color:${colours[bank]}">${bank}</h2>
      <div class="stat">${rows.length}</div>
      <small>Transactions in the selected context</small>
      <p class="note">${
        revealed ? flagged+" labelled suspicious transactions visible"
                 : "Independent bank view — network labels hidden"
      }</p>
      <table>
        <thead><tr><th>Time</th><th>Account</th><th>Amount</th></tr></thead>
        <tbody>${shown.map(t=>`
          <tr class="${revealed && t.role==="ring" ? "flagged":""}">
            <td>${safe(String(t.timestamp).slice(5,16))}</td>
            <td>${safe(
              (t.from_bank===bank?t.from_acct:t.to_acct).slice(0,10)
            )}…</td>
            <td>${safe(Number(t.amount_paid).toLocaleString())}
              <small>${safe(t.currency)}</small></td>
          </tr>`).join("")}</tbody>
      </table>
      <p class="note">Showing ${shown.length} of ${rows.length} rows.
      A cross-bank transfer can appear on two bank screens.</p>`;
    container.appendChild(panel);
  });

  document.getElementById("networkTitle").textContent =
    revealed ? "Cross-bank laundering pattern revealed"
             : "Shared intelligence — currently hidden";

  document.getElementById("summary").textContent =
    revealed
      ? DATA.accounts.length+" linked accounts and "+DATA.ring.length+
        " labelled transactions across three demo banks."
      : "Each bank sees only its own slice of the transactions.";

  const svg = document.getElementById("network");
  svg.style.display = revealed ? "block" : "none";
  if (revealed) drawNetwork(svg);
  drawFreezes();
}

function drawNetwork(svg) {
  svg.innerHTML = "";
  const NS = "http://www.w3.org/2000/svg";

  function element(tag,attributes) {
    let e = document.createElementNS(NS,tag);
    Object.entries(attributes).forEach(([k,v])=>e.setAttribute(k,v));
    svg.appendChild(e);
    return e;
  }

  const bankOf = {};
  DATA.ring.forEach(t=>{
    bankOf[t.from_acct]=t.from_bank;
    bankOf[t.to_acct]=t.to_bank;
  });

  const pos = {};
  DATA.accounts.forEach((a,i)=>{
    let angle = -Math.PI/2 + 2*Math.PI*i/DATA.accounts.length;
    pos[a]={x:375+220*Math.cos(angle),y:245+170*Math.sin(angle)};
  });

  DATA.ring.forEach(t=>{
    const p=pos[t.from_acct], q=pos[t.to_acct];
    if (!p||!q) return;
    const dx=q.x-p.x,dy=q.y-p.y;
    const len=Math.hypot(dx,dy)||1;
    const ux=dx/len,uy=dy/len;
    const x1=p.x+24*ux,y1=p.y+24*uy;
    const x2=q.x-30*ux,y2=q.y-30*uy;
    element("line",{x1,y1,x2,y2,stroke:"#f8ca72",
                    "stroke-width":2.5});
    element("polygon",{
      points:`${x2},${y2} ${x2-12*ux-6*uy},${y2-12*uy+6*ux} ${x2-12*ux+6*uy},${y2-12*uy-6*ux}`,
      fill:"#f8ca72"
    });
  });

  DATA.accounts.forEach((a,i)=>{
    const p=pos[a];
    element("circle",{cx:p.x,cy:p.y,r:26,
                      fill:colours[bankOf[a]]||"#64748b"});
    const n=element("text",{x:p.x,y:p.y+4,"text-anchor":"middle",
                            class:"nodeText"});
    n.textContent=i+1;
    const lab=element("text",{x:p.x,y:p.y+42,
                             "text-anchor":"middle",
                             fill:"#dce7f4","font-size":11});
    lab.textContent=a.slice(0,10)+"…";
  });
}

function drawFreezes() {
  const panel=document.getElementById("freezePanel");
  panel.style.display=revealed?"block":"none";
  const el=document.getElementById("freezeContent");
  if (!DATA.freezes.length) {
    el.innerHTML=`<p class="note">No matching account from this ring
      was found in the current freeze plan. This does not mean the
      optimiser approved or rejected these accounts.</p>`;
    return;
  }
  el.innerHTML=DATA.freezes.map(f=>`
    <div style="border-bottom:1px solid #30445c;padding:12px 0">
      <strong>${safe(f.acct)}</strong>
      <p class="note">Risk score: ${safe(f.risk_score ?? "Unavailable")}</p>
      <p>${safe(f.explanation || f.reason_facts || "No explanation available")}</p>
    </div>`).join("")+
    `<p class="note">Recommendations only. Human approval required.</p>`;
}

draw();
</script>
</body>
</html>
"""

page = page.replace("__PAYLOAD__", payload)

out = ROOT / "golden_hour_demo.html"
out.write_text(page, encoding="utf-8")

print("Demo page created:", out)
print("Flagged transactions:", len(ring))
print("Normal context transactions:", len(normal))
print("Ring accounts:", len(accounts))
print("Matching freeze recommendations:", len(freeze_rows))
print("Open golden_hour_demo.html from the Jupyter file browser.")


In [ ]:

# GOLDEN HOUR — READ-ONLY DATA AND INTEGRATION AUDIT
# Run in a NEW Jupyter notebook CODE CELL.
# This code does not create, overwrite or modify files.

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import hmac

SHARED = Path("/home/ibm-user/shared")
PRIVATE = Path("/home/ibm-user/private")
CLEAN = SHARED / "clean_transactions.csv"

REQUIRED = [
    "tx_id", "timestamp", "from_acct", "to_acct",
    "from_bank", "to_bank", "amount_paid",
    "currency", "payment_format", "is_laundering"
]

def report(name, value):
    print(f"{name}: {value}")

def status(ok):
    return "PASS" if ok else "CHECK / FAIL"

print("=" * 65)
print("GOLDEN HOUR — DATA PIPELINE VERIFICATION")
print("=" * 65)

assert CLEAN.is_file(), "Missing clean_transactions.csv"

c = pd.read_csv(CLEAN, dtype={
    "from_acct": "string",
    "to_acct": "string",
    "from_bank": "string",
    "to_bank": "string"
})

# 1. DATASET STRUCTURE

print("\n[1] DATASET STRUCTURE")

missing_cols = sorted(set(REQUIRED) - set(c.columns))
report("Required columns", status(not missing_cols))
report("Missing columns", missing_cols)
report("Row count", len(c))
report("Extra columns", sorted(set(c.columns) - set(REQUIRED)))

if missing_cols:
    raise ValueError(
        "Required columns missing. Do not continue integration."
    )

accounts = pd.Index(
    pd.concat([c["from_acct"], c["to_acct"]]).dropna().unique()
)

banks = sorted(
    pd.concat([c["from_bank"], c["to_bank"]]).dropna().unique()
)

report("Unique accounts", len(accounts))
report("Bank labels", banks)

critical_missing = c[REQUIRED].isna().sum()

report("Critical missing cells", int(critical_missing.sum()))

print("Missing values by column:")
print(critical_missing.to_string())

# 2. TRANSACTION VALIDITY

print("\n[2] TRANSACTION VALIDITY")

timestamps = pd.to_datetime(c["timestamp"], errors="coerce")
amounts = pd.to_numeric(c["amount_paid"], errors="coerce")
labels = pd.to_numeric(c["is_laundering"], errors="coerce")

report("Invalid timestamps", int(timestamps.isna().sum()))
report("Invalid amounts", int(amounts.isna().sum()))
report("Negative amounts", int((amounts < 0).sum()))
report("Zero amounts", int((amounts == 0).sum()))

report(
    "Invalid laundering labels",
    int((~labels.isin([0, 1])).sum())
)

report(
    "Duplicate tx_id",
    int(c["tx_id"].duplicated().sum())
)

# Duplicate content, excluding generated tx_id.
business_cols = [x for x in REQUIRED if x != "tx_id"]

duplicate_records = int(
    c.duplicated(subset=business_cols).sum()
)

report(
    "Duplicate transaction-content rows",
    duplicate_records
)

print("Note: identical content can occur in valid transactions.")
print("Do not delete duplicates without source verification.")

report(
    "Timestamps in ascending order",
    status(timestamps.is_monotonic_increasing)
)

report("Earliest timestamp", timestamps.min())
report("Latest timestamp", timestamps.max())

report(
    "Flagged laundering transactions",
    int((labels == 1).sum())
)

# 3. BANK AND ACCOUNT CONSISTENCY

print("\n[3] BANK AND ACCOUNT CONSISTENCY")

cross = c["from_bank"].ne(c["to_bank"])

report("Cross-bank transactions", int(cross.sum()))

report(
    "Cross-bank percentage",
    f"{100 * cross.mean():.2f}%"
)

report(
    "Self-transfers",
    int(c["from_acct"].eq(c["to_acct"]).sum())
)

report(
    "Account ID format valid",
    status(
        c["from_acct"].str.fullmatch(
            r"A_[0-9a-f]{12}", na=False
        ).all()
        and c["to_acct"].str.fullmatch(
            r"A_[0-9a-f]{12}", na=False
        ).all()
    )
)

owners = pd.concat([
    c[["from_acct", "from_bank"]].rename(
        columns={
            "from_acct": "acct",
            "from_bank": "bank"
        }
    ),
    c[["to_acct", "to_bank"]].rename(
        columns={
            "to_acct": "acct",
            "to_bank": "bank"
        }
    )
], ignore_index=True)

owner_counts = owners.groupby("acct")["bank"].nunique()

conflicting_owners = int((owner_counts > 1).sum())

report(
    "Accounts assigned to multiple demo banks",
    conflicting_owners
)

report(
    "Account-to-bank consistency",
    status(conflicting_owners == 0)
)

# 4. CURRENCY CHECKS

print("\n[4] CURRENCY CHECKS")

print("Currencies:")
print(c["currency"].value_counts(dropna=False).to_string())

report(
    "Missing currency names",
    int(
        c["currency"].fillna("").str.strip().eq("").sum()
    )
)

report(
    "Missing payment formats",
    int(
        c["payment_format"].fillna("").str.strip().eq("").sum()
    )
)

print("IMPORTANT: Amount totals cannot be added across currencies")
print("without an agreed currency-conversion method.")

# 5. PSEUDONYMISATION CHECK

print("\n[5] PSEUDONYMISATION")

key_path = PRIVATE / "gh_key.txt"
lookup_path = PRIVATE / "code_to_account.csv"

report("Private key file exists", key_path.is_file())
report("Private lookup exists", lookup_path.is_file())

report(
    "Private key absent from shared",
    not (SHARED / "gh_key.txt").exists()
)

report(
    "Private lookup absent from shared",
    not (SHARED / "code_to_account.csv").exists()
)

# Never print the secret key or original account identifiers.

if key_path.is_file() and lookup_path.is_file():
    try:
        key = key_path.read_text().strip().encode()

        lookup = pd.read_csv(
            lookup_path,
            dtype={
                "code": "string",
                "real_key": "string"
            }
        )

        required_lookup = {"code", "real_key"}

        if not required_lookup.issubset(lookup.columns):
            print(
                "HMAC verification: INCOMPLETE (lookup columns)"
            )
        else:
            def recompute(real_key):
                return "A_" + hmac.new(
                    key,
                    str(real_key).encode(),
                    hashlib.sha256
                ).hexdigest()[:12]

            recomputed = lookup["real_key"].map(recompute)

            matches = recomputed.eq(lookup["code"])

            no_collision = (
                lookup["code"].nunique()
                == lookup["real_key"].nunique()
            )

            all_present = accounts.isin(lookup["code"]).all()

            report(
                "Lookup HMAC values match",
                status(bool(matches.all()))
            )

            report(
                "No lookup hash collisions",
                status(no_collision)
            )

            report(
                "All clean accounts in private lookup",
                status(bool(all_present))
            )

            report("HMAC lookup rows", len(lookup))

        del key

    except Exception as exc:
        print("HMAC verification: INCOMPLETE")
        print("Exception type:", type(exc).__name__)

else:
    print("HMAC verification: INCOMPLETE")
    print("Cannot verify against private source files.")

print("SECURITY LIMIT: Folder location alone does not")
print("prove that the key is inaccessible to other users.")

# 6. DOWNSTREAM FILE COMPATIBILITY

print("\n[6] TEAM INTEGRATION FILES")

account_set = set(accounts.astype(str))

candidates = {
    "Alex risk": [
        "account_risk.csv"
    ],
    "Guardian V2": [
        "freeze_plan_candidate.csv",
        "freeze_plan.csv"
    ],
    "Granite explanations": [
        "freeze_plan_explained.csv"
    ],
    "Graph features": [
        "graph_features.csv"
    ],
    "Account labels": [
        "account_labels.csv"
    ]
}

for component, filenames in candidates.items():

    print(f"\n{component}")

    for filename in filenames:

        p = SHARED / filename

        if not p.exists():
            print(f"  {filename}: NOT FOUND")
            continue

        try:
            table = pd.read_csv(p, low_memory=False)

            print(f"  {filename}: FOUND")
            print(f"  Rows: {len(table)}")
            print(f"  Columns: {list(table.columns)}")

            if "acct" in table.columns:

                vals = table["acct"].dropna().astype(str)

                not_found = int(
                    (~vals.isin(account_set)).sum()
                )

                print(
                    f"  Unknown account references: {not_found}"
                )

                print(
                    f"  Unique accounts: {vals.nunique()}"
                )

                print(
                    f"  Duplicate account rows: "
                    f"{vals.duplicated().sum()}"
                )

                print(
                    f"  Account compatibility: "
                    f"{status(not_found == 0)}"
                )

            else:
                print(
                    "  No acct column: inspect schema manually"
                )

            if "risk_score" in table.columns:

                s = pd.to_numeric(
                    table["risk_score"],
                    errors="coerce"
                )

                invalid = int(
                    (s.isna() | ~s.between(0, 1)).sum()
                )

                print(
                    f"  Invalid risk scores: {invalid}"
                )

        except Exception as exc:
            print(
                "  Could not inspect file:",
                type(exc).__name__
            )

# 7. FINAL NOTES

print("\n" + "=" * 65)
print("AUDIT COMPLETE — READ-ONLY")
print("=" * 65)

print("Not proved by this audit:")
print("1. Original timestamps/amounts match source transactions.")
print("2. Alex trained on this exact CSV and final label version.")
print("3. Guardian V2 used Alex's latest risk scores.")
print("4. Model evaluation has no train/test leakage.")
print("5. Key permissions and Jupyter access are secure.")
print("6. End-to-end execution produced the current outputs.")
print("7. Monetary freeze totals are currency-normalised.")

print("\nNo shared files were modified.")


In [ ]:

from pathlib import Path
import pandas as pd

shared = Path("/home/ibm-user/shared")

files = [
    "clean_transactions.csv",
    "graph_features.csv",
    "account_labels.csv",
    "account_risk.csv",
    "freeze_plan_candidate.csv",
    "freeze_plan.csv",
    "freeze_plan_explained.csv",
    "granite_explanation_review.csv",
    "demo_ring.csv",
    "golden_hour_demo.html"
]

print("GOLDEN HOUR — FINAL PRODUCT FILE CHECK")
print("=" * 55)

for name in files:
    path = shared / name
    if path.exists():
        print(f"FOUND    {name}  ({path.stat().st_size:,} bytes)")
    else:
        print(f"MISSING  {name}")

print("\nFINAL FREEZE RECOMMENDATIONS")
print("=" * 55)

for name in [
    "freeze_plan_candidate.csv",
    "freeze_plan.csv",
    "freeze_plan_explained.csv"
]:
    path = shared / name
    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{name}: {len(df)} rows")
        print("Columns:", list(df.columns))
        if "acct" in df.columns:
            print("Unique accounts:", df["acct"].nunique())
        if "explanation" in df.columns:
            valid = df["explanation"].fillna("").astype(str).str.strip().ne("")
            print("Non-empty explanations:", int(valid.sum()))

print("\nCheck completed. No files modified.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

p = Path("/home/ibm-user/shared")

risk = pd.read_csv(p / "account_risk.csv")
freeze = pd.read_csv(p / "freeze_plan.csv")
explained = pd.read_csv(p / "freeze_plan_explained.csv")

print("GOLDEN HOUR — FINAL INTEGRATION CHECK")
print("=" * 50)

# Verify Guardian accounts and risk scores against Alex's output
joined = freeze.merge(
    risk[["acct", "risk_score"]],
    on="acct",
    how="left",
    suffixes=("_freeze", "_alex"),
    validate="one_to_one"
)

unknown = int(joined["risk_score_alex"].isna().sum())
matched = np.isclose(
    joined["risk_score_freeze"],
    joined["risk_score_alex"],
    rtol=1e-6,
    atol=1e-8,
    equal_nan=False
)

print("Guardian accounts missing from Alex:", unknown)
print("Guardian risk scores matching Alex:", int(matched.sum()), "/", len(joined))

# Verify Granite preserved account identities and risk scores
joined2 = freeze.merge(
    explained[["acct", "risk_score", "explanation"]],
    on="acct",
    how="left",
    suffixes=("_freeze", "_granite"),
    validate="one_to_one"
)

risk_matches = np.isclose(
    joined2["risk_score_freeze"],
    joined2["risk_score_granite"],
    rtol=1e-6,
    atol=1e-8,
    equal_nan=False
)

valid_explanations = (
    joined2["explanation"].fillna("")
    .astype(str).str.strip().ne("")
)

print("Granite risk scores matching Guardian:",
      int(risk_matches.sum()), "/", len(joined2))
print("Accounts with explanations:",
      int(valid_explanations.sum()), "/", len(joined2))

print("\nOVERALL")
if unknown == 0 and matched.all() and risk_matches.all() and valid_explanations.all():
    print("PASS — Saved risk scores and account IDs agree across all stages.")
else:
    print("CHECK REQUIRED — One or more downstream outputs disagree.")

print("\nNo files were modified.")


In [ ]:

# GOLDEN HOUR — RISK SCORE MISMATCH DIAGNOSIS
# Read-only: does not modify any files.

import pandas as pd
import numpy as np
from pathlib import Path

p = Path("/home/ibm-user/shared")

alex = pd.read_csv(p / "account_risk.csv")
guardian = pd.read_csv(p / "freeze_plan.csv")

comparison = guardian[
    ["rank", "acct", "risk_score"]
].merge(
    alex[["acct", "risk_score"]],
    on="acct",
    how="left",
    suffixes=("_guardian", "_alex"),
    validate="one_to_one"
)

comparison["difference"] = (
    comparison["risk_score_guardian"] -
    comparison["risk_score_alex"]
)

comparison["absolute_difference"] = (
    comparison["difference"].abs()
)

print("GOLDEN HOUR — RISK SCORE DIAGNOSIS")
print("=" * 70)

print(comparison[
    [
        "rank",
        "acct",
        "risk_score_guardian",
        "risk_score_alex",
        "difference"
    ]
].to_string(index=False))

print("\nSUMMARY")
print("=" * 70)

print("Maximum difference:",
      comparison["absolute_difference"].max())

print("Average difference:",
      comparison["absolute_difference"].mean())

print("Guardian risk range:",
      guardian["risk_score"].min(),
      "to",
      guardian["risk_score"].max())

print("Alex risk range:",
      alex["risk_score"].min(),
      "to",
      alex["risk_score"].max())

print("\nSaved file modification times:")

for name in ["account_risk.csv", "freeze_plan.csv",
             "freeze_plan_explained.csv"]:
    file = p / name
    print(name, pd.Timestamp.fromtimestamp(
        file.stat().st_mtime
    ))

print("\nNo files were modified.")
